In [1]:
#Intention Detection based on the Subcat method

In [ ]:
from __future__ import annotations

import math
import re
from pathlib import Path
from typing import Dict, List, Tuple, Set
from collections import defaultdict, Counter

import pandas as pd

# =========================
# I/O (edit these)
# =========================
BASE_DIR = Path(r"D:\3 - RQ3_2\Intention_3")  # <-- change as needed

INPUT_CSV = BASE_DIR / "All_episodes_with_messages.csv"
DICTIONARY_CSV = BASE_DIR / "dictionary_lemmatized.csv"

# Output: IDF-weighted Subcat + per-row ngram suggestions
OUTPUT_CSV = BASE_DIR / "All_episodes_with_messages_with_intentions_subcat_idf_ngrams.csv"

# Aggregate candidate ngrams across all boundaries
AGG_NGRAMS_CSV = BASE_DIR / "ngram_keyword_candidates_aggregate.csv"

# =========================
# Boost coefficients (equal weights across commit/pr/issue)
# =========================
ISSUE_BOOST = 1.0
COMMIT_BOOST = 1.0
PR_BOOST = 1.0

# =========================
# Episode schema columns
# =========================
END_COMMIT_COL = "episode_end_commit_sha"

START_PARTS: List[Tuple[List[str], float]] = [
    (["start_commit_subject", "start_commit_body"], COMMIT_BOOST),
    (["start_pr_title", "start_pr_body"], PR_BOOST),
    (["start_issue_title"], ISSUE_BOOST),
    (["start_issue_body"], ISSUE_BOOST),
    (["start_issue_comments"], ISSUE_BOOST),
    (["start_issue_summary"], 1.1),  # keep as-is from your snippet; set to 1.0 for strict equality
]

END_PARTS: List[Tuple[List[str], float]] = [
    (["end_commit_subject", "end_commit_body"], COMMIT_BOOST),
    (["end_pr_title", "end_pr_body"], PR_BOOST),
    (["end_issue_title"], ISSUE_BOOST),
    (["end_issue_body"], ISSUE_BOOST),
    (["end_issue_comments"], ISSUE_BOOST),
    (["end_issue_summary"], 1.1),  # keep as-is from your snippet; set to 1.0 for strict equality
]

# =========================
# Scoring / selection params
# =========================
MIN_SCORE = 1.5
MULTI_RATIO = 0.80
MAX_LABELS = 3

CONF_W_SHARE = 0.6
CONF_W_MARGIN = 0.4

HIGH_TH = 0.75
MED_TH = 0.55

# =========================
# Lemmatization (inflectional, offline)
# =========================
_VOWELS = set("aeiou")


def _lemma_alpha(token: str) -> str:
    w = token.lower()
    if len(w) <= 2:
        return w

    irregular = {
        "are": "be",
        "is": "be",
        "was": "be",
        "were": "be",
        "been": "be",
        "being": "be",
        "has": "have",
        "had": "have",
        "having": "have",
        "does": "do",
        "did": "do",
        "done": "do",
        "doing": "do",
    }
    if w in irregular:
        return irregular[w]

    # gerund
    if w.endswith("ying") and len(w) > 5:  # lying->lie, tying->tie
        return w[:-4] + "ie"

    if w.endswith("ing") and len(w) > 5:
        base = w[:-3]
        # running -> run (double consonant)
        if len(base) >= 2 and base[-1] == base[-2] and base[-1] not in _VOWELS:
            base = base[:-1]
        # at/bl/iz -> add e (approx)
        if base.endswith(("at", "bl", "iz")):
            base = base + "e"
        return base

    # past tense
    if w.endswith("ied") and len(w) > 4:
        return w[:-3] + "y"

    if w.endswith("ed") and len(w) > 4:
        base = w[:-2]
        # stopped -> stop
        if len(base) >= 2 and base[-1] == base[-2] and base[-1] not in _VOWELS:
            base = base[:-1]
        return base

    # plurals
    if w.endswith("ies") and len(w) > 4:
        return w[:-3] + "y"

    if w.endswith(("sses", "shes", "ches", "xes", "zes")) and len(w) > 4:
        return w[:-2]  # remove 'es'

    if w.endswith("s") and not w.endswith("ss") and len(w) > 3:
        return w[:-1]

    return w


# =========================
# Tokenization helpers
# =========================
def _safe_str(v) -> str:
    if v is None:
        return ""
    if isinstance(v, float) and pd.isna(v):
        return ""
    s = str(v).strip()
    return "" if s.lower() == "nan" else s


def normalize_text_for_tokens(text: str) -> str:
    text = re.sub(r"([a-z])([A-Z])", r"\1 \2", str(text))  # de-camelcase
    text = text.replace("_", " ").replace("-", " ")
    return text.lower()


def tokenize_and_lemma(text: str) -> List[str]:
    """
    Keeps digits (e2e, v2, aab, etc). Lemmatizes only purely alphabetic tokens.
    """
    if not text:
        return []
    text = normalize_text_for_tokens(text)
    raw = re.findall(r"[a-z0-9]+", text)
    out: List[str] = []
    for t in raw:
        if any(ch.isdigit() for ch in t):
            out.append(t)
        else:
            out.append(_lemma_alpha(t))
    return out


def count_phrase_occurrences(tokens: List[str], phrase_tokens: List[str]) -> int:
    if not phrase_tokens or not tokens:
        return 0
    if len(phrase_tokens) == 1:
        p = phrase_tokens[0]
        return sum(1 for t in tokens if t == p)
    n = len(phrase_tokens)
    cnt = 0
    for i in range(len(tokens) - n + 1):
        if tokens[i : i + n] == phrase_tokens:
            cnt += 1
    return cnt


# =========================
# Dictionary loading (LONG format)
# =========================
def load_dictionary_long(path: Path) -> Tuple[Dict[str, List[Dict]], List[Dict]]:
    """
    Expects columns: label, keyword, optional weight, optional group.
    If label == "Blacklist" (case-insensitive), goes to blacklist.
    """
    df = pd.read_csv(path)

    colmap = {str(c).strip().lower(): c for c in df.columns}
    if "label" not in colmap or "keyword" not in colmap:
        raise ValueError("dictionary.csv must have columns: label, keyword")

    label_col = colmap["label"]
    keyword_col = colmap["keyword"]
    weight_col = colmap.get("weight", None)
    group_col = colmap.get("group", None)

    dict_terms: Dict[str, List[Dict]] = defaultdict(list)
    blacklist_terms: List[Dict] = []

    for _, row in df.iterrows():
        lab = _safe_str(row.get(label_col, "")).strip()
        kw = _safe_str(row.get(keyword_col, "")).strip()
        if not lab or not kw:
            continue

        base_wt = 1.0
        if weight_col is not None:
            try:
                base_wt = float(row.get(weight_col, 1.0))
            except Exception:
                base_wt = 1.0

        grp = _safe_str(row.get(group_col, "")).strip().upper() if group_col is not None else ""

        lemma_tokens = tokenize_and_lemma(kw)
        if not lemma_tokens:
            continue

        item = {
            "keyword": kw,
            "base_weight": base_wt,  # keep original
            "weight": base_wt,       # will be replaced by IDF-weighted later
            "stem_tokens": lemma_tokens,  # name kept for compatibility
            "group": grp,
            "idf": 1.0,
        }

        if lab.lower() == "blacklist":
            blacklist_terms.append(item)
        else:
            dict_terms[lab].append(item)

    # de-dup by token sequence within label
    def dedup(items: List[Dict]) -> List[Dict]:
        seen = set()
        out = []
        for it in items:
            k = " ".join(it["stem_tokens"])
            if k in seen:
                continue
            seen.add(k)
            out.append(it)
        return out

    dict_terms = {lab: dedup(items) for lab, items in dict_terms.items()}
    blacklist_terms = dedup(blacklist_terms)

    if not dict_terms:
        raise ValueError("No label keywords loaded from dictionary.csv")

    return dict_terms, blacklist_terms


# =========================
# Build boundary parts
# =========================
def build_parts_from_row(row: pd.Series, spec: List[Tuple[List[str], float]]) -> List[Tuple[str, float]]:
    parts: List[Tuple[str, float]] = []
    for cols, mult in spec:
        txt = "\n".join([_safe_str(row.get(c, "")) for c in cols]).strip()
        if txt:
            parts.append((txt, mult))
    return parts


# =========================
# Constraints using GLOBAL groups
# =========================
def passes_constraints(label: str, global_groups: Set[str]) -> bool:
    if label == "Introduce / strengthen CI-backed tests":
        return ("CI" in global_groups) and ("TEST" in global_groups)

    if label == "Migrate or modernise CI infrastructure":
        return ("MIGRATE" in global_groups) and ("CI" in global_groups)

    if label == "Clean up or simplify CI / environment configuration":
        return ("CLEANUP" in global_groups) and (("CI" in global_groups) or ("ENV" in global_groups))

    if label == "Address performance or stability issues":
        return (("PERF" in global_groups) or ("FAIL" in global_groups)) and (("CI" in global_groups) or ("TEST" in global_groups))

    return True


# =========================
# Precedence rule
# =========================
def apply_precedence(scores: Dict[str, float], matched_keywords: Dict[str, List[str]], global_groups: Set[str]) -> None:
    perf_lab = "Address performance or stability issues"
    ci_lab = "Introduce / strengthen CI-backed tests"

    has_stability_signal = ("PERF" in global_groups) or ("FAIL" in global_groups)
    has_ci_or_test = ("CI" in global_groups) or ("TEST" in global_groups)

    if has_stability_signal and has_ci_or_test:
        scores[ci_lab] = 0.0
        matched_keywords[ci_lab] = []
        if scores.get(perf_lab, 0.0) > 0:
            scores[perf_lab] *= 1.5


# =========================
# IDF-like weighting (Option C1)
# =========================
def compute_idf_weights(df: pd.DataFrame, dict_terms: Dict[str, List[Dict]], blacklist_terms: List[Dict]) -> None:
    """
    Mutates dict_terms / blacklist_terms in place:
      it["idf"] = log((N+1)/(df+1)) + 1
      it["weight"] = it["base_weight"] * it["idf"]
    where N is the number of boundary-documents (all starts + existing ends).
    """

    def boundary_doc_tokens(row: pd.Series, side: str) -> List[str]:
        if side == "start":
            text = "\n".join(
                [
                    _safe_str(row.get("start_commit_subject", "")),
                    _safe_str(row.get("start_commit_body", "")),
                    _safe_str(row.get("start_pr_title", "")),
                    _safe_str(row.get("start_pr_body", "")),
                    _safe_str(row.get("start_issue_title", "")),
                    _safe_str(row.get("start_issue_body", "")),
                    _safe_str(row.get("start_issue_comments", "")),
                    _safe_str(row.get("start_issue_summary", "")),
                ]
            )
        else:
            text = "\n".join(
                [
                    _safe_str(row.get("end_commit_subject", "")),
                    _safe_str(row.get("end_commit_body", "")),
                    _safe_str(row.get("end_pr_title", "")),
                    _safe_str(row.get("end_pr_body", "")),
                    _safe_str(row.get("end_issue_title", "")),
                    _safe_str(row.get("end_issue_body", "")),
                    _safe_str(row.get("end_issue_comments", "")),
                    _safe_str(row.get("end_issue_summary", "")),
                ]
            )
        return tokenize_and_lemma(text)

    # Build all boundary-documents
    docs: List[List[str]] = []
    for _, row in df.iterrows():
        docs.append(boundary_doc_tokens(row, "start"))
        if not pd.isna(row.get(END_COMMIT_COL)):
            docs.append(boundary_doc_tokens(row, "end"))

    N = len(docs)
    if N == 0:
        return

    # Collect all phrases we need df for
    all_items: List[Dict] = []
    lengths_needed: Set[int] = set()

    for _lab, items in dict_terms.items():
        for it in items:
            all_items.append(it)
            lengths_needed.add(len(it["stem_tokens"]))
    for it in blacklist_terms:
        all_items.append(it)
        lengths_needed.add(len(it["stem_tokens"]))

    lengths_needed = sorted([n for n in lengths_needed if n > 0])

    # Build n-gram "seen" set per doc for those lengths (efficient df counting)
    df_counter: Counter[str] = Counter()

    for toks in docs:
        seen: Set[str] = set()
        L = len(toks)
        for n in lengths_needed:
            if n > L:
                continue
            if n == 1:
                for t in toks:
                    seen.add(t)
            else:
                for i in range(L - n + 1):
                    seen.add(" ".join(toks[i : i + n]))

        # Count df: if phrase appears in doc, increment once
        for it in all_items:
            phrase = " ".join(it["stem_tokens"])
            if phrase in seen:
                df_counter[phrase] += 1

    # Assign IDF-like weights
    for it in all_items:
        phrase = " ".join(it["stem_tokens"])
        df_kw = float(df_counter.get(phrase, 0))
        idf = math.log((N + 1.0) / (df_kw + 1.0)) + 1.0
        it["idf"] = idf
        it["weight"] = float(it.get("base_weight", 1.0)) * idf


# =========================
# N-gram suggestions
# =========================
try:
    from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
    STOP = set(ENGLISH_STOP_WORDS)
except Exception:
    STOP = {
        "the","a","an","and","or","to","of","in","on","for","with","by","is","are","was","were","be",
        "this","that","it","as","at","from"
    }


def suggest_ngrams(tokens: List[str], dict_phrase_set: Set[str], max_n: int = 3, top_k: int = 20) -> List[str]:
    toks = [t for t in tokens if t not in STOP]
    c = Counter()
    L = len(toks)

    for n in range(1, max_n + 1):
        if n == 1:
            for t in toks:
                if len(t) <= 2:
                    continue
                if t in dict_phrase_set:
                    continue
                c[t] += 1
        else:
            for i in range(L - n + 1):
                ng = toks[i : i + n]
                if all(w in STOP for w in ng):
                    continue
                phrase = " ".join(ng)
                if phrase in dict_phrase_set:
                    continue
                c[phrase] += 1

    ranked = sorted(c.items(), key=lambda kv: (kv[1], len(kv[0].split()), len(kv[0])), reverse=True)
    return [p for p, _ in ranked[:top_k]]


# =========================
# Scoring
# =========================
def classify_parts(
    parts: List[Tuple[str, float]],
    dict_terms: Dict[str, List[Dict]],
    blacklist_terms: List[Dict],
    cap_per_keyword_per_part: bool = True,
) -> Tuple[Dict[str, float], Dict[str, List[str]], List[str], Set[str]]:
    """
    Returns:
      scores[label]
      matched_keywords[label]
      blacklist_hits
      global_groups
    """
    token_parts: List[Tuple[List[str], float]] = []
    for text, mult in parts:
        toks = tokenize_and_lemma(text)
        if toks:
            token_parts.append((toks, mult))

    scores = {label: 0.0 for label in dict_terms}
    matched_keywords: Dict[str, List[str]] = {label: [] for label in dict_terms}

    # blacklist hits
    blacklist_hits: List[str] = []
    for it in blacklist_terms:
        for toks, _mult in token_parts:
            if count_phrase_occurrences(toks, it["stem_tokens"]) > 0:
                blacklist_hits.append(it["keyword"])
                break
    blacklist_hits = sorted(set(blacklist_hits))

    global_groups: Set[str] = set()

    # label scoring
    for label, items in dict_terms.items():
        for it in items:
            total_occ = 0.0
            hit = False
            for toks, mult in token_parts:
                occ = count_phrase_occurrences(toks, it["stem_tokens"])
                if occ > 0:
                    hit = True
                    if cap_per_keyword_per_part:
                        occ = 1
                    total_occ += (occ * mult)

            if hit:
                scores[label] += float(it["weight"]) * total_occ
                matched_keywords[label].append(it["keyword"])

                g = (it.get("group") or "").strip().upper()
                if g:
                    global_groups.add(g)

    # Apply constraints
    for label in list(scores.keys()):
        if not passes_constraints(label, global_groups):
            scores[label] = 0.0
            matched_keywords[label] = []

    return scores, matched_keywords, blacklist_hits, global_groups


# =========================
# Selection + confidence
# =========================
def assign_labels_multi(
    scores: Dict[str, float],
    matched_keywords: Dict[str, List[str]],
    blacklist_hits: List[str],
    min_score: float = MIN_SCORE,
    multi_ratio: float = MULTI_RATIO,
    max_labels: int = MAX_LABELS,
) -> Dict[str, object]:
    items = sorted(scores.items(), key=lambda kv: kv[1], reverse=True)
    top_label, top_score = items[0] if items else ("", 0.0)
    second_label, second_score = items[1] if len(items) > 1 else ("", 0.0)

    total = float(sum(scores.values()))
    if top_score < min_score:
        return {
            "label_str": None,
            "top_label": top_label, "top_score": float(top_score),
            "second_label": second_label, "second_score": float(second_score),
            "total_score": float(total),
            "selected_score_sum": 0.0,
            "confidence_share_selected": 0.0,
            "confidence_margin_selected": 0.0,
            "confidence_value": 0.0,
            "confidence_level": "UNLABELED_NOISE" if blacklist_hits else "UNLABELED",
            "matched_keywords": "",
            "blacklist_hits": "; ".join(blacklist_hits),
        }

    chosen: List[str] = []
    chosen_scores: List[float] = []

    for lab, sc in items:
        if sc < min_score:
            break
        if sc >= top_score * multi_ratio:
            chosen.append(lab)
            chosen_scores.append(sc)
        if len(chosen) >= max_labels:
            break

    chosen_set = set(chosen)
    next_unselected = 0.0
    for lab, sc in items:
        if lab not in chosen_set:
            next_unselected = sc
            break

    selected_sum = float(sum(chosen_scores))
    share_selected = (selected_sum / total) if total > 0 else 0.0
    min_selected = float(min(chosen_scores)) if chosen_scores else 0.0
    margin_selected = ((min_selected - next_unselected) / min_selected) if min_selected > 0 else 0.0
    margin_selected = max(0.0, min(1.0, margin_selected))

    confidence_value = (CONF_W_SHARE * share_selected) + (CONF_W_MARGIN * margin_selected)

    if confidence_value >= HIGH_TH and selected_sum >= 2.5:
        level = "HIGH"
    elif confidence_value >= MED_TH:
        level = "MEDIUM"
    else:
        level = "LOW"

    matched = sorted({kw for lab in chosen for kw in matched_keywords.get(lab, [])})

    return {
        "label_str": " || ".join(chosen),
        "top_label": top_label, "top_score": float(top_score),
        "second_label": second_label, "second_score": float(second_score),
        "total_score": float(total),
        "selected_score_sum": float(selected_sum),
        "confidence_share_selected": float(share_selected),
        "confidence_margin_selected": float(margin_selected),
        "confidence_value": float(confidence_value),
        "confidence_level": level,
        "matched_keywords": "; ".join(matched),
        "blacklist_hits": "; ".join(blacklist_hits),
    }


def label_boundary(
    row: pd.Series,
    parts_spec,
    dict_terms,
    blacklist_terms,
    dict_phrase_set: Set[str],
    add_ngram_suggestions: bool = True,
) -> Dict[str, object]:
    parts = build_parts_from_row(row, parts_spec)
    scores, matched_keywords, blacklist_hits, global_groups = classify_parts(parts, dict_terms, blacklist_terms)

    apply_precedence(scores, matched_keywords, global_groups)

    out = assign_labels_multi(scores, matched_keywords, blacklist_hits)

    if add_ngram_suggestions:
        boundary_tokens = tokenize_and_lemma("\n".join([t for t, _m in parts]))
        out["suggested_ngrams"] = "; ".join(suggest_ngrams(boundary_tokens, dict_phrase_set, max_n=3, top_k=20))
    else:
        out["suggested_ngrams"] = ""

    return out


# =========================
# Main
# =========================
def main() -> None:
    if not INPUT_CSV.exists():
        raise FileNotFoundError(f"Input CSV not found: {INPUT_CSV}")
    if not DICTIONARY_CSV.exists():
        raise FileNotFoundError(f"Dictionary CSV not found: {DICTIONARY_CSV}")

    df = pd.read_csv(INPUT_CSV)
    dict_terms, blacklist_terms = load_dictionary_long(DICTIONARY_CSV)

    # IDF-like weighting applied to dictionary/blacklist weights
    compute_idf_weights(df, dict_terms, blacklist_terms)

    # phrase set (for filtering suggested ngrams that are already in dict/blacklist)
    dict_phrase_set: Set[str] = set()
    for _lab, items in dict_terms.items():
        for it in items:
            dict_phrase_set.add(" ".join(it["stem_tokens"]))
    for it in blacklist_terms:
        dict_phrase_set.add(" ".join(it["stem_tokens"]))

    # Aggregate candidates across boundaries
    agg_candidates: Counter[str] = Counter()

    start_results = []
    end_results = []

    for _, row in df.iterrows():
        # start
        s = label_boundary(row, START_PARTS, dict_terms, blacklist_terms, dict_phrase_set, add_ngram_suggestions=True)
        start_results.append(s)
        if s.get("suggested_ngrams"):
            agg_candidates.update([p.strip() for p in s["suggested_ngrams"].split(";") if p.strip()])

        # end
        if pd.isna(row.get(END_COMMIT_COL)):
            end_results.append({
                "label_str": None,
                "top_label": "", "top_score": 0.0,
                "second_label": "", "second_score": 0.0,
                "total_score": 0.0,
                "selected_score_sum": 0.0,
                "confidence_share_selected": 0.0,
                "confidence_margin_selected": 0.0,
                "confidence_value": 0.0,
                "confidence_level": "NO_END_COMMIT",
                "matched_keywords": "",
                "blacklist_hits": "",
                "suggested_ngrams": "",
            })
        else:
            e = label_boundary(row, END_PARTS, dict_terms, blacklist_terms, dict_phrase_set, add_ngram_suggestions=True)
            end_results.append(e)
            if e.get("suggested_ngrams"):
                agg_candidates.update([p.strip() for p in e["suggested_ngrams"].split(";") if p.strip()])

    s_df = pd.DataFrame(start_results).add_prefix("idf_start_")
    e_df = pd.DataFrame(end_results).add_prefix("idf_end_")

    out = pd.concat([df, s_df, e_df], axis=1)
    out.to_csv(OUTPUT_CSV, index=False, encoding="utf-8")
    print("[ok] wrote:", OUTPUT_CSV)

    # Aggregate candidates file
    agg = pd.DataFrame(agg_candidates.most_common(500), columns=["candidate_ngram", "count_across_boundaries"])
    agg.to_csv(AGG_NGRAMS_CSV, index=False, encoding="utf-8")
    print("[ok] wrote:", AGG_NGRAMS_CSV)


if __name__ == "__main__":
    main()


[ok] wrote: D:\3 - RQ3_2\Intention_3\All_episodes_with_messages_with_intentions_subcat.csv


In [ ]:
#information measure measurement: rankning Commit, PR and Issues context based on their level of information